In [ ]:
from tqdm import tqdm  # ilerleme çubuğu için

In [ ]:
image_to_label   = {}
image_to_classes = {}
class_to_images  = defaultdict(list)

def scan_dataset_from_labels(img_dir: Path, lbl_dir: Path, desc=""):
    img_dir = Path(img_dir)
    lbl_dir = Path(lbl_dir)

    exts = [".jpg", ".jpeg", ".png", ".JPG", ".PNG"]
    lbl_files = list(lbl_dir.glob("*.txt"))

    print(f"[SCAN] {desc} -> {len(lbl_files)} label dosyası")

    for lbl_path in tqdm(lbl_files, desc=desc):
        lines = read_label_lines_robust(lbl_path)

        classes = set()
        for line in lines:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cid_str = parts[0].strip()
            try:
                cid = int(cid_str)
            except:
                try:
                    cid = int(float(cid_str))
                except:
                    continue
            classes.add(cid)

        if not classes:
            continue

        stem = lbl_path.stem
        img_path = None
        for ext in exts:
            candidate = img_dir / f"{stem}{ext}"
            if candidate.exists():
                img_path = candidate
                break

        if img_path is None:
            # eşleşen image yoksa geç
            continue

        image_to_label[img_path] = lbl_path
        image_to_classes[img_path] = classes
        for cid in classes:
            class_to_images[cid].append(img_path)


In [ ]:
# önce sözlükleri sıfırla
image_to_label   = {}
image_to_classes = {}
class_to_images  = defaultdict(list)

scan_dataset_from_labels(face_images_dir,   face_labels_dir,   desc="FaceMask")
scan_dataset_from_labels(helmet_images_dir, helmet_labels_dir, desc="Helmet")
scan_dataset_from_labels(belt_images_dir,   belt_labels_dir,   desc="SafetyBelt")
#scan_dataset_from_labels(gog_images_dir,    gog_labels_dir,    desc="SafetyGoggles")
scan_dataset_from_labels(vest_images_dir,   vest_labels_dir,   desc="SafetyVests")

print("Başlangıç sınıf başına etiketli görsel sayıları:")
for cname, cid in FINAL_CLASS_TO_ID.items():
    print(f"{cid} - {cname}: {len(class_to_images[cid])}")


[SCAN] FaceMask -> 809 label dosyası


FaceMask: 100%|██████████| 809/809 [00:06<00:00, 130.56it/s]


[SCAN] Helmet -> 4581 label dosyası


Helmet: 100%|██████████| 4581/4581 [00:48<00:00, 94.00it/s] 


[SCAN] SafetyBelt -> 1705 label dosyası


SafetyBelt: 100%|██████████| 1705/1705 [00:14<00:00, 116.45it/s]


[SCAN] SafetyVests -> 2268 label dosyası


SafetyVests: 100%|██████████| 2268/2268 [00:19<00:00, 118.42it/s]

Başlangıç sınıf başına etiketli görsel sayıları:
0 - helmet: 4580
1 - vest: 2268
2 - safety_belt: 1704
3 - mask: 0


In [ ]:
all_images = list(image_to_classes.keys())
random.shuffle(all_images)

val_ratio = 0.2
val_size  = int(len(all_images) * val_ratio)

val_images       = set(all_images[:val_size])
train_candidates = [img for img in all_images if img not in val_images]

len(all_images), len(train_candidates), len(val_images)


(9361, 7489, 1872)

In [ ]:
aug_counter = 0

def augment_image_pil(img: Image.Image) -> Image.Image:
    if random.random() < 0.7:
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.6, 1.4))
    if random.random() < 0.7:
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.6, 1.4))
    if random.random() < 0.7:
        img = ImageEnhance.Color(img).enhance(random.uniform(0.6, 1.4))
    if random.random() < 0.5:
        arr = np.array(img).astype(np.float32)
        noise = np.random.normal(0, 8, arr.shape)
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
        img = Image.fromarray(arr)
    return img


In [ ]:
def create_augmented_copy(img_path: Path, lbl_path: Path):
    global aug_counter
    aug_counter += 1

    img = Image.open(img_path).convert("RGB")
    aug_img = augment_image_pil(img)

    new_stem = f"{img_path.stem}_aug{aug_counter}"
    new_img_path = img_path.parent / f"{new_stem}{img_path.suffix}"
    aug_img.save(new_img_path)

    new_lbl_path = lbl_path.parent / f"{new_stem}.txt"
    lines = read_label_lines_robust(lbl_path)
    with open(new_lbl_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    classes = set(image_to_classes[img_path])
    image_to_label[new_img_path] = new_lbl_path
    image_to_classes[new_img_path] = classes
    for cid in classes:
        class_to_images[cid].append(new_img_path)

    return new_img_path


In [ ]:
TARGET_PER_CLASS = 1000

# Orijinal görüntü listeleri (augment kaynakları)
orig_class_to_images = {cid: list(imgs) for cid, imgs in class_to_images.items()}

class_counts = {cid: 0 for cid in FINAL_CLASS_TO_ID.values()}
selected_train = set()

random.shuffle(train_candidates)

# 1) Downsample – her görüntüyü, tüm sınıfları 2000'i geçmiyorsa trene al
for img in train_candidates:
    img_classes = image_to_classes[img]
    if not img_classes:
        continue
    if all(class_counts[cid] < TARGET_PER_CLASS for cid in img_classes):
        selected_train.add(img)
        for cid in img_classes:
            class_counts[cid] += 1

print("Downsample sonrası (augment yok):")
for cname, cid in FINAL_CLASS_TO_ID.items():
    print(f"{cname}: {class_counts[cid]}")


KeyError: 4

In [ ]:
# 2) Eksik sınıfları augment ile 2000'e tamamla
for cname, cid in FINAL_CLASS_TO_ID.items():
    if class_counts[cid] >= TARGET_PER_CLASS:
        continue

    needed = TARGET_PER_CLASS - class_counts[cid]
    print(f"\nSınıf {cname} (id={cid}) için augment gerekiyor. Eksik: {needed}")

    # Bu sınıfı içeren, val'de olmayan görüntüler
    sources = [img for img in orig_class_to_images.get(cid, []) if img not in val_images]

    if not sources:
        print("  Uyarı: Bu sınıf için kaynak görüntü yok!")
        continue

    idx = 0
    while class_counts[cid] < TARGET_PER_CLASS:
        img = sources[idx % len(sources)]
        lbl = image_to_label[img]

        new_img_path = create_augmented_copy(img, lbl)
        selected_train.add(new_img_path)

        for c2 in image_to_classes[new_img_path]:
            class_counts[c2] += 1

        idx += 1

print("\nNihai train sınıf sayıları:")
for cname, cid in FINAL_CLASS_TO_ID.items():
    print(f"{cname}: {class_counts[cid]}")

print("Toplam train görsel sayısı:", len(selected_train))
print("Val görsel sayısı:", len(val_images))



Sınıf glasses (id=3) için augment gerekiyor. Eksik: 1000
  Uyarı: Bu sınıf için kaynak görüntü yok!

Sınıf mask (id=4) için augment gerekiyor. Eksik: 380


KeyboardInterrupt: 

In [ ]:
train_list_path = BASE / "train_balanced.txt"
val_list_path   = BASE / "val.txt"

with open(train_list_path, "w") as f:
    for p in sorted(selected_train):
        f.write(str(p) + "\n")

with open(val_list_path, "w") as f:
    for p in sorted(val_images):
        f.write(str(p) + "\n")

print("Balanced train list:", train_list_path)
print("Val list:", val_list_path)


In [ ]:
data_yaml_path = BASE / "data.yaml"

data_yaml = f"""path: {BASE}

train: {train_list_path}
val: {val_list_path}

nc: {len(FINAL_CLASSES)}
names:
"""
for i, cname in enumerate(FINAL_CLASSES):
    data_yaml += f"  {i}: {cname}\n"

with open(data_yaml_path, "w") as f:
    f.write(data_yaml)

print(data_yaml)


In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO


In [ ]:
model = YOLO("yolo11n.pt")  # isterse yolo11s/m/l deneyebilirsin

results = model.train(
    data=str(data_yaml_path),
    imgsz=640,
    epochs=80,
    batch=16,
    workers=2,
    patience=20,
    cos_lr=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
)


In [ ]:
best = YOLO("runs/detect/train/weights/best.pt")

sample_paths = [str(p) for p in list(val_images)[:10]]

best.predict(
    source=sample_paths,
    conf=0.4,
    save=True
)
